# Qwen3-4B LoRA Fine-Tuning
**Runtime → Change runtime type → T4 GPU (free tier)**

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers peft datasets trl bitsandbytes accelerate

In [ ]:
# Step 2: Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Step 3: Load RAG training data
# Upload rag_training_data.jsonl to Colab first
import json

def load_jsonl(path):
    """Load a JSONL file, returns list of dicts."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))
    return data

# RAG task training data (query distiller + chunk scorer)
training_data = load_jsonl("rag_training_data.jsonl")
print(f"Loaded rag_training_data.jsonl: {len(training_data)} samples")

# NOTE: training_data.jsonl (factual Q&A) is NOT used for RAG fine-tuning.
# The vector database stores facts — the model only needs to learn
# how to classify intents, optimize queries, and score chunks.

In [ ]:
# Step 4: Load model with 4-bit quantization (fits in T4 16GB VRAM)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen3-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded!")

In [ ]:
# Step 5: Apply LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 6: Prepare dataset
from datasets import Dataset

def format_chat(item):
    text = (
        f"<|im_start|>user\n{item['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{item['output']}<|im_end|>"
    )
    return {"text": text}

dataset = Dataset.from_list(training_data).map(format_chat)
print(f"Dataset ready: {len(dataset)} samples")
print(f"\nExample:\n{dataset[0]['text'][:300]}...")

In [ ]:
# Step 7: Train
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen3-4b-lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    max_seq_length=1024,
    dataset_text_field="text",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    warmup_steps=5,
    report_to="none",
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Training started...")
trainer.train()
print("Training complete!")

In [ ]:
# Step 8: Save the LoRA adapter
model.save_pretrained("./qwen3-4b-lora/adapter")
tokenizer.save_pretrained("./qwen3-4b-lora/adapter")
print("Adapter saved!")

In [ ]:
# Step 9: Test the fine-tuned model

print("=" * 60)
print("TESTING: Query Distiller")
print("=" * 60)

distiller_tests = [
    '[DISTILL]\nTODAY: 2026-03-08\nMESSAGE: "hello"',
    '[DISTILL]\nTODAY: 2026-03-08\nMESSAGE: "what is DOGE"',
    '[DISTILL]\nTODAY: 2026-03-08\nMESSAGE: "tell me about the latest ICE raids"',
    '[DISTILL]\nTODAY: 2026-03-08\nMESSAGE: "who are you"',
    '[DISTILL]\nTODAY: 2026-03-08\nMESSAGE: "what happened this week with the Iran situation"',
    '[DISTILL]\nTODAY: 2026-03-08\nMESSAGE: "reparations"',
    '[DISTILL]\nTODAY: 2026-03-08\nMESSAGE: "can you summarize everything we talked about"',
]

for test in distiller_tests:
    inputs = tokenizer(f"<|im_start|>user\n{test}<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True)
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    msg = test.split('MESSAGE: "')[1].rstrip('"') if 'MESSAGE: "' in test else test
    print(f"Q: {msg}")
    print(f"A: {response}")
    print("-" * 50)

print("\n" + "=" * 60)
print("TESTING: Chunk Scorer")
print("=" * 60)

scorer_test = """[SCORE]
Query: DOGE

[Chunk 1] (2025-02-15)
The Department of Government Efficiency (DOGE), led by Elon Musk, has canceled over 1,100 federal contracts.

[Chunk 2] (2025-03-18)
Trump's elimination of federal employees could shrink the Black middle class.

[Chunk 3] (2024-11-05)
The 2024 presidential election results showed Donald Trump winning with 312 electoral votes."""

inputs = tokenizer(f"<|im_start|>user\n{scorer_test}<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Scorer output:\n{response}")

In [ ]:
# Step 10: Download adapter to your PC
# Option A: Download as zip
!zip -r qwen3-4b-lora-adapter.zip ./qwen3-4b-lora/adapter
from google.colab import files
files.download('qwen3-4b-lora-adapter.zip')

# Option B: Push to Hugging Face (uncomment below)
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
# model.push_to_hub("your-username/qwen3-4b-lora")
# tokenizer.push_to_hub("your-username/qwen3-4b-lora")